<a href="https://colab.research.google.com/github/ceuratfmg2mai/fakereviews/blob/colab/04.%20manual_data_annotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplicación de la técnica "Anotación Manual de Datos" y "Evaluación de la Fiabilidad Inter-evaluador"

Con el fin de garantizar la calidad y fiabilidad en la evaluación de cómo se clasifican las reseñas, se implementará un proceso de anotación manual de datos, donde un conjunto seleccionado de reseñas será etiquetado por evaluadores humanos como 'Fake' o 'Genuine'. Este proceso tiene como objetivo la creación de un dataset de referencia o "ground truth".  Además, Para validar la consistencia y fiabilidad de estas etiquetas humanas, y en nuestro caso donde han participado múltiples anotadores, se aplicará una Evaluación de la Fiabilidad Inter-evaluador, utilizando la métrica Kappa de Cohen. Este análisis cuantitativo del acuerdo entre evaluadores le usaremos para garantizar que el ground truth es objetivo, proporcionando así una base para la posterior evaluación del rendimiento del sistema de clasificación automática (DeepSeek) y para contextualizar los resultados de la búsqueda por similitud en Qdrant.

Además establecemos un Kappa con el siguiente baremo:

* Moderado (0.41-0.60).
* Sustancial (0.61-0.80)
* Casi Perfecto (0.81-1.00)

# Librerías

In [103]:
import polars as pl
import pandas as pd
import pyarrow
import matplotlib.pyplot as plt
plt.style.use('Solarize_Light2')
import seaborn as sns
import time
start_time_global = time.time()
import os
import sys
import random
from google.colab import userdata
from google.colab import drive

import sklearn
from sklearn.metrics import cohen_kappa_score

color = '\033[1m\033[38;5;208m'
print(f"{color}Versión pandas: {pd.__version__}")
print(f"{color}Versión polars: {pl.__version__}")
print(f"{color}Versión pyarrow: {pyarrow.__version__}")
print(f"{color}Versión sklearn: {sklearn.__version__}")

Versión pandas: 2.2.2
Versión polars: 1.21.0
Versión pyarrow: 18.1.0
Versión sklearn: 1.6.1


In [104]:
drive.mount('/content/drive', force_remount=True)
explicit_work_path = '/content/drive/MyDrive/Colab Notebooks/tfm_grupo_2/'
print(os.listdir(explicit_work_path))
# Agrega la carpeta fakereviews al syspath del proyecto
if explicit_work_path not in sys.path:
    sys.path.append(explicit_work_path)

Mounted at /content/drive
['base_reviews.json', 'yelp_academic_dataset_review_selected_0_255.jsonl', 'final_reviews_categorize_0_255.jsonl', 'data_yelp', 'yelp_academic_dataset_review_selected.jsonl', 'yelp_academic_dataset_review_selected.arrow', 'yelp_academic_dataset_review_selected.csv', 'yelp_academic_dataset_review_selected_prompt.arrow', 'yelp_academic_dataset_review_selected_prompt.jsonl', 'final_reviews_categorize.jsonl', 'yelp_academic_dataset_review_final_selected_1900.jsonl', 'yelp_academic_dataset_review_golden_evaluador_1.jsonl', 'yelp_academic_dataset_review_golden_evaluador_2.json', 'ground_truth.jsonl']


# Desarrollo

El desarrollo de este notebook estará estructurado de la siguiente forma:

1. Lectura de los dataset anotados por humanos.
2. Seguido de, extracción de datos relevates para su comparación.
4. Finalmente, realizamos la evalucaión de fiabilidad de los datos.

## 1. Lectura de los dataset anotados por humanos
En este apartado, hacemos la lectura de los ficheros anotados por humanos

### Evaluador 1

La técnica aplicada por este evaluador considera:

1.    La sentimentalidad de la frase y la categoría del establecimiento (ej: hotel, restaurant), en caso de realizar una reseña a un "Hotel" el contexto deberá ir acorde.
2.    Seguidamente, las estrellas totales dadas por el usuario, junto con su votación.  En caso de una alta votación y una sola votación, más la evaluación del tercer punto.
3.    Se tiene encuenta, la media de estrellas del establecimiento, junto con la votación del usuario.

Dependiendo de la evaluación de los puntos anteriores sería Fake o Genuine.
Ejemplo:
* Un "Spa" con una reseña de "Limpieza de coches", más, una unica votación de 5 estrellas y una media del establecimiento de votación de 2, será conciderada "Fake."
* Un "Spa" con una reseña acorde al establecimiento, más, una unica votción de 5 estrellas y una media del establecimiento de 4, será conciderada "Genuine".



In [105]:
# Lectura del fichero de las reseñas anotadas por el evaluador 1
review_categorice_file_path = f'{explicit_work_path}yelp_academic_dataset_review_golden_evaluador_1.jsonl'
df_data_reviews_evaluador_uno_pl = pl.read_json(review_categorice_file_path)
print(f"\nTotal de reseñas anotadas por el evaluador 1: {df_data_reviews_evaluador_uno_pl.shape[0]}")
print("-" * 20)
display(df_data_reviews_evaluador_uno_pl['classification'].value_counts())


Total de reseñas anotadas por el evaluador 1: 200
--------------------


classification,count
str,u32
"""Fake""",54
"""Genuine""",146


### Evaluador 2

La técnica aplicada por este evaluador considera:

1. Para usuarios con alto numero de reseñas y con palabras y vocabulario normal - Genuine.
2. Para usuarios con muchas reseñas y con palabras y vocabulario exagerado - Fake.
* (best, amazing, awesome, love, favorite, incredible,........ para puntuaciones de 5)
* (horrible, worst, never, awful, terrible, waste...... para puntuaciones de 1)
3. Para usuarios con pocas reseñas y con palabras y vocabulario exagerado - Fake.
* (best, amazing, awesome, love, favorite, incredible,........ para puntuaciones de 5)
* (horrible, worst, never, awful, terrible, waste...... para puntuaciones de 1)

In [106]:
# Lectura del fichero de las reseñas anotadas por el evaluador 1
review_categorice_file_path = f'{explicit_work_path}yelp_academic_dataset_review_golden_evaluador_2.json'
df_data_reviews_evaluador_dos_pl = pl.read_json(review_categorice_file_path)
print(f"\nTotal de reseñas anotadas por el evaluador 1: {df_data_reviews_evaluador_dos_pl.shape[0]}")
print("-" * 20)
display(df_data_reviews_evaluador_dos_pl['classification'].value_counts())


Total de reseñas anotadas por el evaluador 1: 100
--------------------


classification,count
str,u32
"""Genuine""",84
"""Fake""",16


# Extracción de datos relevates para su comparación.

Obtenemos los datos de las columnas relevantes

In [107]:
df_eval1 = df_data_reviews_evaluador_uno_pl.select(["review_id", "classification"])
df_eval1 = df_eval1.rename({"classification": "clase_eval1"})

In [108]:
df_eval2 = df_data_reviews_evaluador_dos_pl.select(["review_id", "classification"])
df_eval2 = df_eval2.rename({"classification": "clase_eval2"})

Unimos los dos conjuntos de datos, pero solo mantendremos las filas donde 'review_id' esté presente en ambos DataFrames

In [109]:
df_comparacion = df_eval1.join(df_eval2, on="review_id", how="inner")

A continuación, realizamos la evaluación Kappa

In [110]:
# --- Extraer las listas de etiquetas para el cálculo de Kappa ---
if not df_comparacion.is_empty() and df_comparacion.height > 1: # Necesitas al menos 2 para Kappa con sentido
    etiquetas_eval1 = df_comparacion.get_column('clase_eval1').to_list()
    etiquetas_eval2 = df_comparacion.get_column('clase_eval2').to_list()

    print("\n--- Etiquetas para Kappa (solo reseñas comunes) ---")
    print(f"Evaluador 1: {etiquetas_eval1}")
    print(f"Evaluador 2: {etiquetas_eval2}")

    # --- Calcular Kappa de Cohen ---
    kappa = cohen_kappa_score(etiquetas_eval1, etiquetas_eval2)

    print(f"\n--- Fiabilidad Inter-evaluador ---")
    print(f"Kappa de Cohen: {kappa:.4f}")

    # Interpretación (igual que en el ejemplo anterior)
    if kappa < 0: interpretacion = "Pobre acuerdo."
    elif kappa == 0: interpretacion = "Acuerdo equivalente al azar."
    elif kappa < 0.21: interpretacion = "Ligero acuerdo."
    elif kappa < 0.41: interpretacion = "Aceptable acuerdo."
    elif kappa < 0.61: interpretacion = "Moderado acuerdo."
    elif kappa < 0.81: interpretacion = "Sustancial acuerdo."
    else: interpretacion = "Casi perfecto o perfecto acuerdo."
    print(f"Interpretación: {interpretacion}")

    acuerdo_simple_porcentaje = sum(1 for a, b in zip(etiquetas_eval1, etiquetas_eval2) if a == b) / len(etiquetas_eval1) * 100
    print(f"Porcentaje de acuerdo simple (observado): {acuerdo_simple_porcentaje:.2f}%")

else:
    print("\nNo hay suficientes reseñas comunes entre los dos evaluadores para calcular Kappa de forma significativa.")
    if not df_comparacion.is_empty():
        print(f"Número de reseñas comunes: {df_comparacion.height}")


--- Etiquetas para Kappa (solo reseñas comunes) ---
Evaluador 1: ['Genuine', 'Genuine', 'Genuine', 'Genuine', 'Fake', 'Fake', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Fake', 'Fake', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Fake', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Fake', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Fake', 'Genuine', 'Genuine', 'Fake', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Genuine', 'Fake', 'Genuine', 'Genuine', 'Fake', 'Fake', 'Genuine', 'Genuine', 'Genuine', 'Genuine'

En las siguientes reseñas no hubo acuerdo de evaluación.

In [111]:
desacuerdos = df_comparacion.filter(
    pl.col('clase_eval1') != pl.col('clase_eval2')
)
if not desacuerdos.is_empty():
    print("\n--- Reseñas con Desacuerdo entre Evaluadores ---")
    display(desacuerdos) # O selecciona columnas específicas


--- Reseñas con Desacuerdo entre Evaluadores ---


review_id,clase_eval1,clase_eval2
str,str,str
"""y85Ncu7U5blel7n8e5yPwg""","""Genuine""","""Fake"""
"""zkXuEndr0XUH-PTsMQ8TRA""","""Genuine""","""Fake"""
"""EE5pgRFOSKbTSxS-I-Z9tg""","""Genuine""","""Fake"""
"""pCDBhuLCVGiCmgXgD76GCg""","""Genuine""","""Fake"""
"""FW5bHPexLQjz8s51zv2h0w""","""Genuine""","""Fake"""


A pesar de los esfuerzos el acuerdo entre evaluadores (medido por Kappa o el porcentaje de acuerdo simple) no ha alcanzado el nivel deseado para todos los ítems, y necesitas proceder con un único conjunto de etiquetas "ground truth" para evaluar a DeepSeek o para otros análisis.

In [112]:
# Método 2: Usando apply (menos idiomático en Polars para esto, pero muestra la lógica)
def elegir_aleatoriamente(row_tuple):
  # row_tuple será una tupla de (valor_clase_eval1, valor_clase_eval2)
  return random.choice(row_tuple)

etiquetas_finales = []
for row in df_comparacion.select(["clase_eval1", "clase_eval2"]).iter_rows():
    etiquetas_finales.append(random.choice(row))
df_con_etiqueta_final = df_comparacion.with_columns(
    pl.Series("etiqueta_final_aleatoria", etiquetas_finales)
)

In [113]:
df_dataset_final = df_con_etiqueta_final.select([
    "review_id",
    "etiqueta_final_aleatoria"
])
df_dataset_final = df_dataset_final.rename({"etiqueta_final_aleatoria": "classification"})
print("\n--- Dataset Final con una Etiqueta por Reseña (seleccionada aleatoriamente) ---")
print(df_dataset_final)


--- Dataset Final con una Etiqueta por Reseña (seleccionada aleatoriamente) ---
shape: (100, 2)
┌────────────────────────┬────────────────┐
│ review_id              ┆ classification │
│ ---                    ┆ ---            │
│ str                    ┆ str            │
╞════════════════════════╪════════════════╡
│ 78CkRZ7RTAzHSWj8T6Cwfg ┆ Genuine        │
│ EqmRuP57et9yflKKqNIe9w ┆ Genuine        │
│ 7KJ-aQamSonJWK7-7ynUvA ┆ Genuine        │
│ texyymRcA9bpve4OdmHD1g ┆ Genuine        │
│ Eh812QNPErEFDaXXgvpwww ┆ Fake           │
│ …                      ┆ …              │
│ TSq8Y6IQMrgvEd7K_I-znA ┆ Genuine        │
│ 2Nq-8xz28cPAOK-1O4xxDQ ┆ Genuine        │
│ FW5bHPexLQjz8s51zv2h0w ┆ Genuine        │
│ 1JC-ByVuokz6hscgIvcl7Q ┆ Genuine        │
│ ZXgEQtk0mT6bU-SVOm_9Xg ┆ Genuine        │
└────────────────────────┴────────────────┘


Guardamos los datos seleccionados

In [114]:
try:
    review_selected_file_path = f'{explicit_work_path}ground_truth.jsonl'
    df_dataset_final.write_json(review_selected_file_path)
except Exception as e:
    print(f"Error al guardar jsonl: {e}")
else:
    print('Fichero seleccionados guardado en JSONL')

Fichero seleccionados guardado en JSONL


# Conclusión

Considerando el acuerdo sustancial alcanzado entre los evaluadores humanos, reflejado en un Kappa de Cohen de 0.7871 y un porcentaje de acuerdo simple del 95.00%, se ha establecido una base de anotaciones manuales de alta fiabilidad. Para construir el conjunto de datos final de "ground truth" utilizado en las subsecuentes etapas de evaluación y con el fin de asegurar una representación imparcial en los escasos casos de discrepancia entre evaluadores, se procedió a una selección aleatoria de una de las dos evaluaciones asignadas a cada reseña que fue valorada por ambos. Este método pragmático para la consolidación de etiquetas garantizará que cada reseña cuente con una única clasificación definitiva ('Fake' o 'Genuine'), permitiendo así una evaluación consistente y robusta del modelo DeepSeek y de los análisis posteriores.

# Información de la sesión

In [115]:
!pip install session_info > /dev/log 2>&1

In [116]:
import session_info
session_info.show(html=False)

-----
google              NA
matplotlib          3.10.0
pandas              2.2.2
polars              1.21.0
pyarrow             18.1.0
seaborn             0.13.2
session_info        v1.0.1
sklearn             1.6.1
-----
IPython             7.34.0
jupyter_client      6.1.12
jupyter_core        5.7.2
notebook            6.5.7
-----
Python 3.11.12 (main, Apr  9 2025, 08:55:54) [GCC 11.4.0]
Linux-6.1.123+-x86_64-with-glibc2.35
-----
Session information updated at 2025-05-18 18:53


In [117]:
print(f'Total ejecución {time.time() - start_time_global:.2f} segundos')

Total ejecución 6.63 segundos
